# Gap 3: Per-Diagram Nesting Decomposition

**Previous tests** applied $N_\lambda$ uniformly: every $n$-loop diagram weighted by $\lambda^{-n}$.
This is equivalent to coupling rescaling $\alpha \to \alpha/\lambda$.

**This gap** tests whether weighting diagrams by their *nesting depth* $\sigma_{\rm nest}$
(number of nested self-energy insertions, NOT loop order) gives a structurally
different and potentially useful result.

Key distinction:
- **Loop-order attenuation**: all 2-loop diagrams get weight $\lambda^{-2}$
- **Nesting-depth attenuation**: skeleton 2-loop diagrams ($\sigma_{\rm nest}=0$) keep full weight;
  SE-insertion 2-loop diagrams ($\sigma_{\rm nest}=1$) get weight $\lambda^{-1}$

This IS a different function of $\lambda$ — not a coupling rescaling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.graded_scn import (
    ALPHA_OVER_PI, A_E_EXPERIMENT, A_E_UNCERTAINTY,
    QED_G2_COEFFICIENTS, compute_g2_standard,
    ALPHA_S_MZ, M_Z, M_TAU,
    beta0_standard, alpha_s_running_1loop,
    r_tau_coefficients, r_tau_tree,
    R_TAU_EXPERIMENT, R_TAU_UNCERTAINTY,
    r_ratio_coefficients,
)

# Known 2-loop QED decomposition (Feynman gauge, on-shell)
SE_INSERTION_TOTAL = 0.7714      # I(a)+I(b)+I(c) + counterterms
C2_TOTAL = QED_G2_COEFFICIENTS[2]["value"]
SKELETON_TOTAL = C2_TOTAL - SE_INSERTION_TOTAL

print("=== QED 2-Loop Per-Diagram Decomposition ===")
print(f"SE insertions (sigma_nest=1):  {SE_INSERTION_TOTAL:+.4f} (alpha/pi)^2")
print(f"Skeleton (sigma_nest=0):       {SKELETON_TOTAL:+.4f} (alpha/pi)^2")
print(f"Total C_2:                     {C2_TOTAL:+.6f} (alpha/pi)^2")
print()

# Three C_2(lambda) functions
lambdas = np.linspace(0.5, 5.0, 500)
c2_loop = C2_TOTAL / lambdas**2
c2_nest = SE_INSERTION_TOTAL / lambdas + SKELETON_TOTAL

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.axhline(C2_TOTAL, color='gray', ls='--', lw=1, label=f'Standard C_2={C2_TOTAL:.4f}')
ax1.axhline(SKELETON_TOTAL, color='green', ls=':', lw=1, label=f'Skeleton limit={SKELETON_TOTAL:.4f}')
ax1.plot(lambdas, c2_loop, 'b-', lw=2, label='Loop-order: C_2/lambda^2')
ax1.plot(lambdas, c2_nest, 'r-', lw=2, label='Nesting: SE/lambda + skel')
ax1.set_xlabel('lambda'); ax1.set_ylabel('C_2(lambda)')
ax1.set_title('C_2 under different attenuation schemes')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

ax2.plot(lambdas, c2_nest - c2_loop, 'k-', lw=2)
ax2.axhline(0, color='gray', ls='--')
ax2.set_xlabel('lambda'); ax2.set_ylabel('C2_nest - C2_loop')
ax2.set_title('Structural difference between schemes')
ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("\nC_2 values at selected lambda:")
print(f"{'lam':>6} {'Loop-att':>10} {'Nest-att':>10} {'Diff':>10}")
for lam in [1.0, 1.5, 2.0, 3.0, 5.0]:
    cl = C2_TOTAL / lam**2
    cn = SE_INSERTION_TOTAL / lam + SKELETON_TOTAL
    print(f"{lam:6.1f} {cl:10.4f} {cn:10.4f} {cn-cl:10.4f}")
print("\nAs lambda->inf: loop-order->0, nesting->skeleton limit.")
print("Nesting-depth attenuation is NOT equivalent to coupling rescaling.")

In [ ]:
# Impact on a_e: nesting attenuation at 2-loop only
# (higher loops: no decomposition data available)

a_e_std = compute_g2_standard(5)
sigma_std = (a_e_std - A_E_EXPERIMENT) / A_E_UNCERTAINTY

# Even a tiny change to C_2 is visible at enormous sigma
delta_c2 = SE_INSERTION_TOTAL * (1 - 1/2.0)  # change from lam=1 to lam=2
delta_ae = delta_c2 * ALPHA_OVER_PI**2

print(f"Standard a_e:     {a_e_std:.15e}")
print(f"Experiment:       {A_E_EXPERIMENT:.15e}")
print(f"Standard sigma:   {sigma_std:+.2f}")
print()
print(f"C_2 coefficient of (alpha/pi)^2:")
print(f"  (alpha/pi)^2 = {ALPHA_OVER_PI**2:.4e}")
print(f"  C_2 term = {C2_TOTAL * ALPHA_OVER_PI**2:.4e}")
print(f"  Exp. uncertainty = {A_E_UNCERTAINTY:.4e}")
print(f"  |C_2 term| / uncertainty = {abs(C2_TOTAL * ALPHA_OVER_PI**2) / A_E_UNCERTAINTY:.0f} sigma")
print()
print(f"At lam=2, delta_C2 = {delta_c2:.4f}, delta_ae = {delta_ae:.4e}")
print(f"This is {abs(delta_ae)/A_E_UNCERTAINTY:.0f} sigma deviation.")
print()
print("VERDICT: QED g-2 is measured to part-per-trillion precision.")
print("The 2-loop coefficient is constrained at ~10^6 sigma level.")
print("ANY modification to C_2 -- nesting or loop-order -- produces")
print("enormous deviations. Per-diagram attenuation is structurally")
print("different but equally fatal for QED.")

In [ ]:
# QCD: parametric bubble-chain decomposition
# Model: fraction f of K_n from bubble chains with max nesting depth (n-1)
# K_n^eff(f, lam) = (1-f)*K_n + f*K_n*lam^{-(n-1)}  for n>=2
# K_1^eff = K_1 (1-loop: no self-energy insertions)

K = r_tau_coefficients()
nf = 3
b0_std = beta0_standard(nf)
alpha_s_mtau = alpha_s_running_1loop(M_TAU**2, nf, b0_std)
a_s = alpha_s_mtau / np.pi

print(f"alpha_s(m_tau) = {alpha_s_mtau:.4f}, a_s = alpha_s/pi = {a_s:.5f}, beta_0 = {b0_std:.1f}")
print()

def r_tau_perdiag(f, lam, max_order=4):
    """R_tau with per-diagram nesting attenuation on bubble chains."""
    delta = K[1] * a_s  # n=1: no bubble chains
    for n in range(2, max_order + 1):
        k_eff = (1 - f) * K[n] + f * K[n] * lam**(-(n - 1))
        delta += k_eff * a_s**n
    return r_tau_tree() * (1 + delta)

def r_tau_loop_att(lam, max_order=4):
    """R_tau with loop-order attenuation (coupling rescaling)."""
    delta = sum(K[n] * (a_s / lam)**n for n in range(1, max_order + 1))
    return r_tau_tree() * (1 + delta)

# 2D scan: best lambda for each f
lam_grid = np.linspace(0.5, 10.0, 5000)
print("Best lambda for each bubble-chain fraction f:")
print(f"{'f':>5} {'lam_opt':>7} {'R_tau':>8} {'sigma':>8}")

for f in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    if f == 0.0:
        r_std = r_tau_perdiag(0, 1.0)
        sig = (r_std - R_TAU_EXPERIMENT) / R_TAU_UNCERTAINTY
        print(f"{f:5.1f} {'N/A':>7} {r_std:8.4f} {sig:+8.1f}")
        continue
    residuals = np.array([abs(r_tau_perdiag(f, l) - R_TAU_EXPERIMENT) for l in lam_grid])
    best_idx = np.argmin(residuals)
    best_lam = lam_grid[best_idx]
    r_fit = r_tau_perdiag(f, best_lam)
    sigma = (r_fit - R_TAU_EXPERIMENT) / R_TAU_UNCERTAINTY
    print(f"{f:5.1f} {best_lam:7.2f} {r_fit:8.4f} {sigma:+8.1f}")

print(f"\nExperiment: R_tau = {R_TAU_EXPERIMENT} +/- {R_TAU_UNCERTAINTY}")

In [ ]:
# Structural comparison: per-diagram vs coupling rescaling
print("Structural comparison: per-diagram (f=1) vs coupling rescaling")
print("=" * 65)
print()
print("Coupling rescaling: delta = Sum K_n (a/lam)^n = Sum K_n a^n lam^{-n}")
print("Per-diagram (f=1): delta = K1*a + Sum_{n>=2} K_n a^n lam^{-(n-1)}")
print("                        = K1*a + lam * Sum_{n>=2} K_n (a/lam)^n")
print()
print("Key: K_1 is NOT attenuated in per-diagram scheme!")
print()

print(f"{'lam':>5}  {'Loop-att d':>12}  {'PerDiag d':>12}  {'delta_diff':>12}  {'R_tau diff':>10}")
for lam in [1.0, 1.1, 1.2, 1.5, 2.0, 3.0]:
    delta_loop = sum(K[n] * (a_s/lam)**n for n in range(1, 5))
    delta_pd = K[1]*a_s + sum(K[n] * a_s**n * lam**(-(n-1)) for n in range(2, 5))
    r_loop = r_tau_tree() * (1 + delta_loop)
    r_pd = r_tau_tree() * (1 + delta_pd)
    print(f"{lam:5.1f}  {delta_loop:12.6f}  {delta_pd:12.6f}  {delta_pd-delta_loop:12.6f}  {r_pd-r_loop:10.4f}")

print()

# Cross-observable: fit R_tau then predict R(10.5 GeV)
print("Cross-observable test: fit R_tau, predict R(sqrt_s=10.5 GeV)")
print("-" * 60)
r_coeffs = r_ratio_coefficients(4)  # nf=4
alpha_s_10 = alpha_s_running_1loop(10.52**2, 4, beta0_standard(4))
a_10 = alpha_s_10 / np.pi
charges = [2/3, 1/3, 1/3, 2/3]
R0_10 = 3 * sum(q**2 for q in charges)

def r_10_perdiag(f, lam):
    delta = r_coeffs[1] * a_10
    for n in range(2, 5):
        k_eff = (1-f)*r_coeffs[n] + f*r_coeffs[n]*lam**(-(n-1))
        delta += k_eff * a_10**n
    return R0_10 * (1 + delta)

r_10_std = r_10_perdiag(0, 1.0)
print(f"Standard R(10.5 GeV): {r_10_std:.6f}, alpha_s = {alpha_s_10:.4f}")
print()
print(f"{'f':>5} {'lam_opt(Rtau)':>12} {'R(10.5) pred':>14} {'Delta_R':>10} {'Delta_R/R':>10}")
for f in [0.3, 0.5, 0.7, 1.0]:
    res = [abs(r_tau_perdiag(f, l) - R_TAU_EXPERIMENT) for l in lam_grid]
    lam_opt = lam_grid[np.argmin(res)]
    r10_pred = r_10_perdiag(f, lam_opt)
    dr = r10_pred - r_10_std
    print(f"{f:5.1f} {lam_opt:12.2f} {r10_pred:14.6f} {dr:10.6f} {dr/r_10_std:10.2e}")

print()
print("At high energies (small alpha_s), per-diagram corrections are")
print("negligible. Two free parameters fit 1 number with no predictive power.")

## Conclusions

Per-diagram nesting attenuation IS structurally different from coupling rescaling:
it preserves the 1-loop term while differentially weighting higher-order
diagram classes by nesting depth.

However:

1. **QED**: The 2-loop coefficient $C_2$ is constrained at $\sim 10^6\sigma$ precision.
   ANY modification — nesting or loop-order — produces enormous deviations.

2. **QCD**: Without known per-diagram $K_n^{(d)}$ decompositions, the bubble-chain
   fraction $f$ is a free parameter. Two free parameters $(f, \lambda)$ trivially
   fit one observable ($R_\tau$) with no independent prediction.

3. **Cross-observable**: The correction vanishes at high energies (small $\alpha_s$),
   so no testable prediction propagates beyond $m_\tau$.

**Verdict**: Conceptually interesting but empirically empty.